# Interact with the DEX via the `icp` CLI

This notebook walks through the core DEX flows using the [`icp`](https://cli.internetcomputer.org/) CLI only:

1. Discover listed trading pairs
2. Approve the DEX as an ICRC-2 spender on a token ledger
3. Deposit tokens
4. Place a limit order
5. Check order status
6. Withdraw

> **Version note.** Commands target the candid interface shipped in `canister/dex.did` on this branch. If the live staging canister is running an earlier version, a call may fail with a decode error until the canister is upgraded.

## Prerequisites

- [`icp` CLI](https://cli.internetcomputer.org/) installed and on `PATH`
- Launch this notebook from the **project root** so `--environment staging` resolves the `dex` canister name (defined in `icp.yaml`)
- An identity configured via `icp identity new` or `icp identity import` — every `deposit`, `add_limit_order`, and `withdraw` runs as the currently-selected identity, and the DEX keys its internal balances off that caller principal

## Your identity

Queries don't require signing, but updates do. Confirm which identity you're about to act as:

In [ ]:
!icp identity default
!icp identity principal

## 1. List trading pairs

`get_trading_pairs` is a query — cheap, no signing. It returns every listed pair along with:

- `tick_size` — price must be a positive multiple of this value
- `lot_size` — quantity must be a positive multiple of this value (base-token units)

In [ ]:
!icp canister call dex get_trading_pairs '()' --environment staging --query

## 2. Pick a pair

Copy the base and quote ledger principals from the output above and set them below. The values below are placeholders — adjust them to match the pair you picked.

`DEX` is the staging canister ID (also resolved by the `dex` name when `--environment staging` is passed).

In [ ]:
BASE_LEDGER  = "ryjl3-tyaaa-aaaaa-aaaba-cai"   # adjust to your chosen pair
QUOTE_LEDGER = "mxzaz-hqaaa-aaaar-qaada-cai"   # adjust to your chosen pair
DEX          = "proc5-daaaa-aaaar-qb5va-cai"   # staging DEX canister

TICK_SIZE = 10          # copy from the `tick_size` field above
LOT_SIZE  = 1_000_000   # copy from the `lot_size`  field above

## 3. Approve the DEX as an ICRC-2 spender

`deposit` moves tokens by calling `icrc2_transfer_from` on the token ledger, so the DEX must first be approved to spend on your behalf.

Two fees are charged to your on-ledger balance:
1. `icrc2_approve` charges the ledger fee once
2. `icrc2_transfer_from` (triggered later by `deposit`) charges the ledger fee again, on top of the amount transferred

To deposit `N`, approve **at least `N + ledger_fee`** and hold **at least `N + 2 × ledger_fee`** on the ledger.

Check your on-ledger balance and the ledger fee:

In [ ]:
!icp token "$BASE_LEDGER" balance --network ic
!icp canister call "$BASE_LEDGER" icrc1_fee '()' --network ic --query

Now approve — the call below runs against the base ledger directly, so it uses `--network ic` instead of `--environment staging`:

In [ ]:
deposit_amount = 1_000_000   # base-token units
ledger_fee     = 10_000      # replace with the value returned by icrc1_fee above

APPROVE_ARGS = (
    f'(record {{ '
    f'spender = record {{ owner = principal "{DEX}"; subaccount = null }}; '
    f'amount = {deposit_amount + ledger_fee} : nat '
    f'}})'
)
print(APPROVE_ARGS)

!icp canister call "$BASE_LEDGER" icrc2_approve '$APPROVE_ARGS' --network ic

## 4. Deposit

Credits `amount` base tokens to your on-DEX *free* balance and pulls `amount + ledger_fee` out of your ledger account.

In [ ]:
DEPOSIT_ARGS = (
    f'(record {{ '
    f'token_id = record {{ ledger_id = principal "{BASE_LEDGER}" }}; '
    f'amount = {deposit_amount} : nat '
    f'}})'
)
print(DEPOSIT_ARGS)

!icp canister call dex deposit '$DEPOSIT_ARGS' --environment staging

## 5. Check your on-DEX balance

The DEX tracks balance per `(caller, token)`:

- `free` — available for new orders or withdrawal
- `reserved` — locked by open orders

In [ ]:
BALANCE_ARGS = f'(record {{ ledger_id = principal "{BASE_LEDGER}" }})'

!icp canister call dex get_balance '$BALANCE_ARGS' --environment staging --query

## 6. Place a limit order

- A **sell** order locks `quantity` base tokens: `free` → `reserved`
- A **buy** order locks `price × quantity` quote tokens (requires a prior deposit of the quote token — repeat steps 3–4 for `$QUOTE_LEDGER`)

`price` must be a positive multiple of `TICK_SIZE`; `quantity` a positive multiple of `LOT_SIZE`.

Submission is asynchronous: the call returns an order ID immediately, the matching engine processes it on the next canister tick.

In [ ]:
price    = 100        # must be a positive multiple of TICK_SIZE
quantity = LOT_SIZE   # must be a positive multiple of LOT_SIZE

ORDER_ARGS = (
    f'(record {{ '
    f'pair = record {{ base = principal "{BASE_LEDGER}"; quote = principal "{QUOTE_LEDGER}" }}; '
    f'side = variant {{ Sell }}; '
    f'price = {price} : nat64; '
    f'quantity = {quantity} : nat '
    f'}})'
)
print(ORDER_ARGS)

!icp canister call dex add_limit_order '$ORDER_ARGS' --environment staging

## 7. Check order status

Paste the 32-char hex order ID returned above:

In [ ]:
ORDER_ID = "paste-the-order-id-here"

STATUS_ARGS = f'("{ORDER_ID}")'
!icp canister call dex get_order_status '$STATUS_ARGS' --environment staging --query

| Status     | Meaning                                   |
|------------|-------------------------------------------|
| `Pending`  | Accepted, awaiting the matching engine    |
| `Open`     | Resting in the order book                 |
| `Filled`   | Fully filled                              |
| `Canceled` | Canceled                                  |
| `NotFound` | Unknown order ID                          |

## 8. Withdraw

Debits `amount` from your on-DEX *free* balance and sends `amount − ledger_fee` to your principal on the ledger. Only `free` funds are eligible; `reserved` funds (locked by open orders) aren't withdrawable until the order fills or is canceled.

In [ ]:
withdraw_amount = 500_000

WITHDRAW_ARGS = (
    f'(record {{ '
    f'token_id = record {{ ledger_id = principal "{BASE_LEDGER}" }}; '
    f'amount = {withdraw_amount} : nat '
    f'}})'
)

!icp canister call dex withdraw '$WITHDRAW_ARGS' --environment staging

## What's next

- Inspect the append-only event log via `get_events` — every state change (listings, deposits, orders) is recorded. See `canister/dex.did` for the full schema.
- Add the quote-token side of the deposit flow to place buy orders (repeat steps 3–4 targeting `$QUOTE_LEDGER`).
- See `integration_tests/` for end-to-end scenarios that exercise every endpoint programmatically.